# 2. Edge Detection Pipeline — NAIP COG tile

This notebook demonstrates how `edge_detector.py` works. It loads a mock NAIP COG tile, extracts a 2-meter buffer around a polygon boundary, then runs the full edge detection pipeline.

**Pipeline steps:**
1. Contrast enhancement (clip 2–98 percentile)
2. K-means clustering (n=3)
3. Active contour snake

**Visualised outputs:**
- Original image
- Contrast-enhanced image
- K-means clustered labels
- Snake result overlaid
- Deviation overlay

## Imports & setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), "..", "backend"))

from edge_engine.edge_detector import (
    enhance_contrast,
    cluster_pixels,
    compute_edge_gradient,
    active_contour_snake,
    measure_deviation,
    compute_spectral_contrast,
)

print("All imports successful.")

## 2.1 Generate a mock NAIP image patch

We simulate a 4-band NAIP tile (R, G, B, NIR) containing a bright field region with a darker road boundary running diagonally. In production this data would come from a real COG tile via `rasterio`.

In [ ]:
np.random.seed(42)
H, W = 128, 128

# Simulated background: farmland (greenish NIR)
farmland = np.random.normal(loc=(80, 110, 60, 150), scale=15, size=(H, W, 4)).clip(0, 255).astype(np.uint8)

# Simulated road: darker, lower NIR
mask_road = np.zeros((H, W), dtype=bool)
for i in range(H):
    for j in range(W):
        if abs(j - (i + 20)) < 6:
            mask_road[i, j] = True

road_pixels = np.random.normal(loc=(45, 35, 30, 60), scale=8, size=(H, W, 4)).clip(0, 255).astype(np.uint8)

image = np.where(mask_road[..., None], road_pixels, farmland)

print(f"Image shape: {image.shape} (H, W, bands)")
print(f"Data type:   {image.dtype}")
print(f"Pixel range: {image.min()} – {image.max()}")

## 2.2 Extract a 2-metre buffer around the polygon boundary

We simulate a polygon and buffer it. In production the buffer extraction happens via `shapely` before raster masking.

In [ ]:
from shapely.geometry import Polygon, box

# Simulated field polygon (in pixel coordinates for demo)
polygon = Polygon([(20, 20), (108, 20), (108, 108), (20, 108)])
buffered = polygon.buffer(2)  # 2-metre buffer in pixel units for demo

# Create a rectangular crop around the buffered polygon
bounds = [int(b) for b in buffered.bounds]
x_min, y_min, x_max, y_max = bounds
buffer_pixels = image[y_min:y_max, x_min:x_max].copy()

print(f"Buffer crop bounds (pixel coords): {bounds}")
print(f"Buffer crop shape: {buffer_pixels.shape}")

## 2.3 Display original & contrast-enhanced image

`enhance_contrast` does a 2–98 percentile clip and rescale to 0–255.

In [ ]:
# Work with RGB (first 3 bands) for display
rgb = buffer_pixels[..., :3]
enhanced_rgb = enhance_contrast(rgb.astype(float))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(rgb)
axes[0].set_title("Original RGB")
axes[0].axis("off")

axes[1].imshow((enhanced_rgb / enhanced_rgb.max()).clip(0, 1))
axes[1].set_title("Contrast Enhanced (2–98 pctl)")
axes[1].axis("off")

fig.tight_layout()
plt.show()

print(f"Original pixel range: {rgb.min():.1f} – {rgb.max():.1f}")
print(f"Enhanced pixel range: {enhanced_rgb.min():.1f} – {enhanced_rgb.max():.1f}")

## 2.4 K-means clustering (n=3)

`cluster_pixels` runs sklearn KMeans on the full multi-spectral buffer (all 4 bands) and returns a 2-D label array. The three classes typically correspond to *road / soil / vegetation*.

In [ ]:
labels = cluster_pixels(buffer_pixels, n_clusters=3)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(labels, cmap="tab10", interpolation="nearest")
ax.set_title("K-means Clusters (k=3)")
ax.axis("off")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.show()

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} pixels")

## 2.5 Edge gradient

`compute_edge_gradient` applies a Sobel filter to the clustered labels to highlight transitions between classes.

In [ ]:
gradient = compute_edge_gradient(labels)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(gradient, cmap="inferno")
ax.set_title("Edge Gradient (Sobel on clusters)")
ax.axis("off")
plt.show()

## 2.6 Active contour (snake) fitting

`active_contour_snake` fits a snake to the strongest edges. We seed it with a circle centred in the buffer crop to simulate an ML-predicted boundary.

In [ ]:
# Build an initial contour (a circle)
from skimage import color

s = labels.shape
cx, cy = s[1] // 2, s[0] // 2
r = min(s) // 3
angles = np.linspace(0, 2 * np.pi, 80)
initial_contour = np.column_stack([
    cy + r * np.sin(angles),
    cx + r * np.cos(angles),
])

gray = color.rgb2gray(rgb.astype(float))
snake = active_contour_snake(gray, initial_contour, alpha=0.01, beta=0.1, max_iterations=100)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(gray, cmap="gray")
ax.plot(initial_contour[:, 1], initial_contour[:, 0], "--r", linewidth=1.5, label="Initial")
ax.plot(snake[:, 1], snake[:, 0], "-y", linewidth=2, label="Snake")
ax.set_title("Active Contour Snake")
ax.axis("off")
ax.legend()
plt.show()

print(f"Initial contour vertices: {initial_contour.shape[0]}")
print(f"Snake vertices:           {snake.shape[0]}")

## 2.7 Deviation overlay

`measure_deviation` computes per-vertex Euclidean distance between the ML-predicted boundary (initial contour) and the snake-fitted boundary.

In [ ]:
mean_dev, distances = measure_deviation(initial_contour, snake)

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    initial_contour[:, 1], initial_contour[:, 0],
    c=distances, cmap="hot", s=20, edgecolors="white", linewidth=0.3,
)
ax.set_title(f"Deviation Overlay (mean = {mean_dev:.3f} px)")
ax.axis("off")
plt.colorbar(scatter, ax=ax, shrink=0.8, label="Deviation (px)")
plt.show()

print(f"Mean deviation: {mean_dev:.4f} pixels")
print(f"Max deviation:  {distances.max():.4f} pixels")

## 2.8 Spectral contrast

`compute_spectral_contrast` measures inter-cluster distance as a confidence signal.

In [ ]:
contrast = compute_spectral_contrast(buffer_pixels.astype(float), labels)
print(f"Spectral contrast: {contrast:.4f}  (0.0–1.0, higher = clearer edge)")

## 2.9 How `edge_detector.py` works

1. **`enhance_contrast`** — clips image intensity at the 2nd and 98th percentiles, then rescales to full 0–255 range.
2. **`cluster_pixels`** — flattens the multi-spectral buffer, runs sklearn `KMeans(n_clusters=3)`, and reshapes labels back to 2D.
3. **`compute_edge_gradient`** — applies a Sobel filter on the clustered labels.
4. **`active_contour_snake`** — wraps `skimage.segmentation.active_contour`; converts to grayscale if needed.
5. **`measure_deviation`** — builds a `cKDTree` on snake vertices and queries per-vertex distance.
6. **`compute_spectral_contrast`** — computes mean spectral vector per cluster, then the max pairwise Euclidean distance, normalised to [0, 1].

In [ ]:
print("edge_detector.py source:")
with open("../backend/edge_engine/edge_detector.py") as f:
    print(f.read())